# 02 - Linear Regression Baseline

Linear Regression is used as a baseline because it is simple, fast, and easy to interpret.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
DATA_DIR = Path('../data')
MODEL_DIR = Path('../outputs/models')
RESULT_DIR = Path('../outputs/results')
FIG_DIR = Path('../outputs/figures')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

math_df = pd.read_csv(DATA_DIR / 'student-mat.csv', sep=';')
por_df = pd.read_csv(DATA_DIR / 'student-por.csv', sep=';')
math_df['subject'] = 'Math'
por_df['subject'] = 'Portuguese'
df = pd.concat([math_df, por_df], ignore_index=True).drop_duplicates()

In [ ]:
X = df.drop(columns=['G3'])
y = df['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame([{
    'Model': 'Linear Regression',
    'RMSE': rmse,
    'MAE': mae,
    'R2': r2
}])

display(metrics)
metrics.to_csv(RESULT_DIR / 'linear_regression_metrics.csv', index=False)
joblib.dump(model, MODEL_DIR / 'linear_regression_model.pkl')

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel('Actual G3')
plt.ylabel('Predicted G3')
plt.title('Linear Regression: Actual vs Predicted')
plt.show()